# SQL Analysis: U.S. Interest Rates and Housing Market

The goal of this analysis is to use SQL to examine relationships between U.S. interest rates and housing market activity from 2000 to 2026. I will use SQL queries to summarize the data, compare housing activity across interest rate environments, and identify trends over time.

This analysis builds on the exploratory data analysis performed in Python.

In [16]:
import sqlite3
import pandas as pd
from pathlib import Path

In [17]:
processed_data = Path("../data/processed")

In [18]:
df = pd.read_csv(
    processed_data / "merged_housing_interest_rates_2000_2026.csv"
)

df["date"] = pd.to_datetime(df["date"])

In [19]:
conn = sqlite3.connect("../data/housing.db")

In [20]:
df.to_sql("housing_data", conn, index=False, if_exists="replace")

320

# 1. Data Validation

Before running the analysis, I checked that the dataset was loaded into SQLite correctly and that the number of rows matched the original dataset.

In [21]:
row_count = pd.read_sql_query(
    "SELECT COUNT(*) AS row_count FROM housing_data",
    conn
)

print("Rows loaded:", row_count.iloc[0, 0])

Rows loaded: 320


In [22]:
query = """
SELECT *
FROM housing_data
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)
display(result)

,date,fed_funds_rate,mortgage_rate,housing_starts,new_home_sales
0,2000-01-01 00:00:00,5.45,8.21,1636,873
1,2000-02-01 00:00:00,5.73,8.33,1737,856
2,2000-03-01 00:00:00,5.85,8.24,1604,900
3,2000-04-01 00:00:00,6.02,8.15,1626,841
4,2000-05-01 00:00:00,6.27,8.52,1575,857
5,2000-06-01 00:00:00,6.53,8.29,1559,793
6,2000-07-01 00:00:00,6.54,8.15,1463,887
7,2000-08-01 00:00:00,6.50,8.03,1541,848
8,2000-09-01 00:00:00,6.52,7.91,1507,912
9,2000-10-01 00:00:00,6.51,7.80,1549,933


In [23]:
print(df.shape)
print(conn.execute("SELECT COUNT(*) FROM housing_data").fetchone())

(320, 5)
(320,)


# 2. Summary Statistics

I calculated the average interest rates, housing starts, and new home sales across the full study period to get an overall picture of the data.

In [24]:
query = """
SELECT
    ROUND(AVG(fed_funds_rate), 2) AS avg_fed_funds_rate,
    ROUND(AVG(mortgage_rate), 2) AS avg_mortgage_rate,
    ROUND(AVG(housing_starts), 2) AS avg_housing_starts,
    ROUND(AVG(new_home_sales), 2) AS avg_new_home_sales
FROM housing_data;
"""

display(pd.read_sql_query(query, conn))

,avg_fed_funds_rate,avg_mortgage_rate,avg_housing_starts,avg_new_home_sales
0,2.05,5.24,1308.97,695.36


# 3. Comparing High- and Low-Rate Periods

I divided the months into high- and low-rate groups based on the median federal funds rate, then compared average housing starts and new home sales between the two groups.

In [25]:
query = """
WITH median_rate AS (
    SELECT AVG(fed_funds_rate) AS median_value
    FROM (
        SELECT
            fed_funds_rate,
            ROW_NUMBER() OVER (ORDER BY fed_funds_rate) AS rn,
            COUNT(*) OVER () AS total
        FROM housing_data
    )
    WHERE rn IN ((total + 1) / 2, (total + 2) / 2)
)
SELECT
    CASE
        WHEN h.fed_funds_rate > m.median_value THEN 'High Rate'
        ELSE 'Low Rate'
    END AS rate_group,
    COUNT(*) AS months,
    ROUND(AVG(h.housing_starts), 2) AS avg_housing_starts,
    ROUND(AVG(h.new_home_sales), 2) AS avg_new_home_sales
FROM housing_data h
CROSS JOIN median_rate m
GROUP BY rate_group;
"""

display(pd.read_sql_query(query, conn))

,rate_group,months,avg_housing_starts,avg_new_home_sales
0,High Rate,160,1495.16,806.10
1,Low Rate,160,1122.77,584.61


# 4. Annual Trends

I grouped the monthly observations by year to examine how interest rates and housing market activity changed over time.

In [26]:
query = """
SELECT
    SUBSTR(date, 1, 4) AS year,
    ROUND(AVG(fed_funds_rate), 2) AS avg_fed_funds_rate,
    ROUND(AVG(mortgage_rate), 2) AS avg_mortgage_rate,
    ROUND(AVG(housing_starts), 2) AS avg_housing_starts,
    ROUND(AVG(new_home_sales), 2) AS avg_new_home_sales
FROM housing_data
GROUP BY SUBSTR(date, 1, 4)
ORDER BY year;
"""

yearly_results = pd.read_sql_query(query, conn)
display(yearly_results)

,year,avg_fed_funds_rate,avg_mortgage_rate,avg_housing_starts,avg_new_home_sales
0,2000,6.24,8.06,1573.33,880.25
1,2001,3.89,6.97,1601.17,907.25
2,2002,1.67,6.54,1710.25,976.17
3,2003,1.13,5.82,1853.75,1090.67
4,2004,1.35,5.84,1949.50,1200.83
5,2005,3.21,5.87,2072.92,1278.92
6,2006,4.96,6.41,1811.92,1049.33
7,2007,5.02,6.34,1341.83,768.67
8,2008,1.93,6.04,900.00,482.17
9,2009,0.16,5.04,554.00,374.25


# 5. Coverage Check

I checked the number of years included in the analysis and confirmed the first and last years covered by the dataset.

In [27]:
query = """
WITH yearly_data AS (
    SELECT
        SUBSTR(date, 1, 4) AS year,
        AVG(fed_funds_rate) AS fed_funds_rate,
        AVG(mortgage_rate) AS mortgage_rate,
        AVG(housing_starts) AS housing_starts,
        AVG(new_home_sales) AS new_home_sales
    FROM housing_data
    GROUP BY SUBSTR(date, 1, 4)
)
SELECT
    COUNT(*) AS years_analyzed,
    MIN(year) AS first_year,
    MAX(year) AS last_year
FROM yearly_data;
"""

display(pd.read_sql_query(query, conn))

,years_analyzed,first_year,last_year
0,27,2000,2026


In [28]:
yearly_results.to_csv(
    "../data/processed/yearly_housing_interest_rates.csv",
    index=False
)

print("Yearly SQL results saved.")

Yearly SQL results saved.


In [29]:
conn.close()

## Key Findings

- The average federal funds rate was 2.05%, while the average 30-year mortgage rate was 5.24%.
- The dataset contained 160 months in each rate group.
- The high-rate group had higher average housing starts (1,495.16) and new home sales (806.10) than the low-rate group (1,122.77 and 584.61, respectively).
- Annual averages show that interest rates and housing market activity changed substantially over the study period.

## Limitations

These findings describe relationships in the data and do not establish causation. Housing starts and new home sales are reported as seasonally adjusted annual rates (SAAR), in thousands. The 2026 data covers January through August only. Other factors, including housing affordability, housing supply, and broader economic conditions, may also influence housing activity.